# 08-01 BatchNorm 深入理解：为什么要把中间结果拉回稳定范围

Batch Normalization 通常简称 BatchNorm 或 BN。

它不是一个新的网络层用来“增加表达能力”，而是一个让训练更稳定的工具。

先记一句话：

```text
BatchNorm 想解决的是：深层网络训练时，每一层看到的数据分布总是在变，导致后面的层很难稳定学习。
```

这一节我们不从公式开场，而是从“为什么深层网络会不稳定”开始。

## 1. 先想一个普通训练过程

神经网络是一层接一层的。

前一层的输出，就是后一层的输入。

可以写成：

$$
\mathbf{h}^{(1)}=f_1(\mathbf{x})
$$

$$
\mathbf{h}^{(2)}=f_2(\mathbf{h}^{(1)})
$$

$$
\mathbf{h}^{(3)}=f_3(\mathbf{h}^{(2)})
$$

第 2 层要学习的对象，其实是第 1 层输出出来的东西。

第 3 层要学习的对象，又是第 2 层输出出来的东西。

## 2. 问题：前面的层一直在变

训练过程中，每一层的参数都在不断更新。

第 1 层参数一变，它输出的结果也会变。

第 2 层接收到的输入就跟着变。

这会导致一个问题：

```text
后面的层刚适应前面的输入分布，前面的层又变了。
```

这就像你在学习一个规则，但题目的格式一直变。

不是不能学，而是学起来更慢、更不稳定。

## 3. 什么叫输入分布变了

分布这个词听起来抽象，其实可以先理解成：

```text
一批数大概集中在哪里，波动有多大。
```

比如某一层原来接收到的输入大概是：

```text
集中在 0 附近，数值大多在 -1 到 1 之间
```

训练几步之后，前面的层变了，它接收到的输入可能变成：

```text
集中在 20 附近，数值大多在 10 到 30 之间
```

这就是输入分布变了。

对后面的层来说，它面对的学习环境一直在换。

## 4. 为什么分布乱会影响训练

神经网络里有激活函数。

如果输入值太大或太小，Sigmoid、Tanh 这类函数会进入饱和区，梯度变得很小。

如果输入分布忽大忽小，后面的层就会遇到这些问题：

```text
有时输入太大，激活函数进入不舒服的区域
有时输入尺度变化很大，梯度也跟着不稳定
有时需要不断重新适应前一层的新输出
```

所以 BatchNorm 的动机就出现了：

```text
能不能让每一层看到的输入保持在比较稳定、比较舒服的范围？
```

## 5. BatchNorm 的核心动作

BatchNorm 做的第一件事是标准化。

它会在一个 mini-batch 内，计算某一层输出的均值和方差。

均值表示这批数大概集中在哪里。

方差表示这批数波动有多大。

然后 BatchNorm 把这批数变成：

```text
均值大约为 0
方差大约为 1
```

这就像把每一层的中间结果拉回到一个比较稳定的范围。

## 6. 第一步：计算 batch 均值

假设一个 mini-batch 里有 m 个样本。

某个神经元在这 m 个样本上的输出是：

$$
x_1,x_2,\ldots,x_m
$$

先计算均值：

$$
\mu_B=\frac{1}{m}\sum_{i=1}^{m}x_i
$$

这里的 B 表示 batch。

这个均值回答的是：

```text
这一小批数据大概集中在什么位置？
```

## 7. 第二步：计算 batch 方差

接着计算方差：

$$
\sigma_B^2=\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
$$

方差回答的是：

```text
这批数据离均值有多分散？
```

如果方差很大，说明数据波动大。

如果方差很小，说明数据都挤在一起。

BatchNorm 需要方差，是为了知道该用多大的尺度把数据拉回标准范围。

## 8. 第三步：标准化

有了均值和方差，就可以标准化：

$$
\hat{x}_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

这个公式可以分三步读。

第一步：减去均值。

$$
x_i-\mu_B
$$

意思是把数据中心挪到 0 附近。

第二步：除以标准差。

$$
\sqrt{\sigma_B^2+\epsilon}
$$

意思是把数据波动缩放到差不多的尺度。

第三步：epsilon 是安全垫，防止分母为 0。

标准化之后，这批数据大致变成：

```text
中心在 0 附近
波动尺度在 1 附近
```

## 9. 只标准化会不会太死板

会。

如果 BatchNorm 只把数据强行变成均值 0、方差 1，可能会限制网络表达能力。

因为有些层可能确实需要输出不在标准范围内。

所以 BatchNorm 又加了两个可学习参数。

一个负责缩放：

$$
\gamma
$$

一个负责平移：

$$
\beta
$$

最终输出是：

$$
y_i=\gamma\hat{x}_i+\beta
$$

这一步的意思是：

```text
先把数据拉回稳定范围，再让网络自己决定要不要缩放或平移。
```

## 10. 完整 BatchNorm 公式

BatchNorm 的完整过程可以写成四步。

第一步，计算 batch 均值：

$$
\mu_B=\frac{1}{m}\sum_{i=1}^{m}x_i
$$

第二步，计算 batch 方差：

$$
\sigma_B^2=\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
$$

第三步，标准化：

$$
\hat{x}_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
$$

第四步，缩放和平移：

$$
y_i=\gamma\hat{x}_i+\beta
$$

先不要死背公式，先记住每一步在干什么：

```text
算中心 -> 算波动 -> 拉回标准范围 -> 允许网络再调整
```

## 11. gamma 和 beta 不是优化器里的 beta

这里很容易混。

BatchNorm 里的 beta 是一个可学习的平移参数。

优化器里也有 beta，比如 Momentum、RMSProp、Adam 里的 beta，它们通常表示指数移动平均里的历史保留比例。

它们名字一样，但含义完全不同。

可以这样区分：

| 符号 | 在哪里 | 作用 |
| --- | --- | --- |
| BatchNorm 的 gamma | BatchNorm 层 | 控制输出缩放 |
| BatchNorm 的 beta | BatchNorm 层 | 控制输出平移 |
| 优化器的 beta | Momentum / RMSProp / Adam | 控制历史信息保留多少 |

所以看到 beta 时，一定要看它在哪个上下文里。

## 12. BatchNorm 放在哪里

常见顺序是：

```text
Linear / Conv -> BatchNorm -> Activation
```

也就是先做线性变换或卷积，再做 BatchNorm，最后过激活函数。

为什么经常放在激活函数前？

因为我们通常希望把进入激活函数前的值拉回比较合适的范围。

比如 Sigmoid、Tanh 如果输入太大或太小，就容易进入饱和区。

BatchNorm 可以帮助输入值更集中在激活函数比较容易传梯度的区域。

## 13. 训练阶段和推理阶段为什么不同

训练时，BatchNorm 使用当前 mini-batch 的均值和方差。

也就是：

```text
这一批数据来了，就用这一批数据自己的统计量。
```

但推理时可能一次只输入一个样本。

如果只用一个样本计算均值和方差，就很不稳定。

所以训练过程中，BatchNorm 会维护一个移动平均统计量。

推理阶段就使用训练期间积累下来的整体均值和方差。

一句话：

```text
训练时用当前 batch 的统计量；推理时用训练阶段积累的统计量。
```

## 14. BatchNorm 为什么能加快训练

BatchNorm 常常能让训练更快、更稳定。

原因不是它让模型变得更聪明，而是它让每一层面对的数据环境更稳定。

这样后面的层不用一直重新适应前面层输出的剧烈变化。

它还可能让我们使用稍微大一点的学习率。

因为中间激活值被控制在相对稳定的范围内，训练不那么容易发散。

所以可以这样记：

```text
BatchNorm 像给每一层的输入做稳定器，让后面的层更容易学。
```

## 15. BatchNorm 和输入标准化有什么区别

输入标准化通常是对原始数据做的。

比如把图片像素、表格特征先处理成均值接近 0、方差接近 1。

BatchNorm 是对网络中间层的输出做的。

区别可以这样记：

| 方法 | 处理对象 | 什么时候做 |
| --- | --- | --- |
| 输入标准化 | 原始输入数据 | 进入网络前 |
| BatchNorm | 网络中间激活值 | 网络训练过程中 |

输入标准化只能让第一层舒服一点。

BatchNorm 是让中间很多层也尽量舒服一点。

## 16. BatchNorm 和 Dropout 的区别

BatchNorm 和 Dropout 都可能缓解过拟合，但它们主业不同。

Dropout 的核心是随机关闭神经元，减少依赖。

BatchNorm 的核心是稳定中间数据分布，让训练更顺。

可以这样对比：

| 方法 | 主要目标 | 怎么做 |
| --- | --- | --- |
| Dropout | 缓解过拟合 | 随机关闭神经元 |
| BatchNorm | 稳定训练 | 标准化中间激活值 |

BatchNorm 有时会带来一点正则化效果，因为每个 batch 的统计量带有随机性。

但它最主要的作用仍然是稳定训练。

## 17. BatchNorm 的局限

BatchNorm 很常用，但不是所有情况都适合。

它依赖 mini-batch 的统计量。

如果 batch size 很小，当前 batch 的均值和方差可能不稳定。

比如 batch size 只有 1 或 2，那这一小批数据不一定能代表整体分布。

这时候 BatchNorm 效果可能变差。

所以在一些小 batch 场景中，可能会考虑 LayerNorm、GroupNorm 等其他归一化方法。

这些后面进入更深网络时再展开。

## 18. 本节总结

BatchNorm 的逻辑链是：

```text
深层网络中，前面层一直更新
-> 后面层看到的输入分布也一直变
-> 训练变得不稳定
-> BatchNorm 在 mini-batch 内计算均值和方差
-> 先把中间结果标准化
-> 再用 gamma 和 beta 让网络保留表达能力
-> 训练阶段用当前 batch 统计量
-> 推理阶段用训练中积累的整体统计量
```

先记住一句话：

```text
BatchNorm 不是让模型直接变强，而是让每一层的学习环境更稳定。
```

下一节就可以继续进入 MLP 结构设计，或者开始从 MLP 过渡到 CNN。